# M5 - Liquidity and Budget Channel

In [3]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

def find_project_root():
    search = Path(os.getcwd()).resolve()
    for _ in range(6):
        if (search / 'data' / 'processed').exists():
            return search
        search = search.parent
    raise FileNotFoundError("data/processed not found")

PROJECT_ROOT = find_project_root()
PROCESSED = PROJECT_ROOT / 'data' / 'processed'

print(f"Project root: {PROJECT_ROOT}")
print(f"Data path: {PROCESSED}")

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d2e',
    'axes.edgecolor': '#3a3d52',
    'axes.labelcolor': '#c8ccd8',
    'xtick.color': '#8890a4',
    'ytick.color': '#8890a4',
    'text.color': '#c8ccd8',
    'grid.color': '#2a2d3e',
    'grid.alpha': 0.5,
})

CRISIS_PERIODS = [
    ('2014-12-01', '2015-02-28', '#e74c3c', 'Crisis 2014-15'),
    ('2022-02-24', '2022-05-31', '#e67e22', 'Sanctions Feb 2022'),
]

def shade_crises(ax):
    for s, e, color, _ in CRISIS_PERIODS:
        ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), color=color, alpha=0.13)

print("Imports OK")


Project root: C:\Users\Huawei\python_programming\psb_case
Data path: C:\Users\Huawei\python_programming\psb_case\data\processed
Imports OK


In [4]:
def read_csv(name, date_col='date'):
    df = pd.read_csv(PROCESSED / name)
    df[date_col] = pd.to_datetime(df[date_col], format='%d-%m-%Y')
    return df.sort_values(date_col).reset_index(drop=True)

df = read_csv('m5_dataset.csv')
roskazna = read_csv('roskazna_treasury_deposits.csv', date_col='auction_date')

print(f"M5: {len(df):,} rows | {df.date.min().date()} - {df.date.max().date()}")
print(f"Roskazna: {len(roskazna):,} auctions")
df.head(3)


M5: 3,077 rows | 2014-02-03 - 2026-05-08
Roskazna: 2,289 auctions


,date,budget_funds_date,budget_funds_total_mln_rub,federal_budget_funds_mln_rub,regional_local_budget_funds_mln_rub,other_budget_funds_mln_rub,extra_budgetary_funds_mln_rub,roskazna_auctions_count,roskazna_max_volume_mln_rub,roskazna_demand_volume_mln_rub,...,secured_loans_standing_bln_rub,cbr_liabilities_standard_instruments_bln_rub,deposit_auctions_bln_rub,deposit_standing_bln_rub,cobr_bln_rub,nonstandard_refundable_operations_bln_rub,correspondent_accounts_bln_rub,required_reserves_avg_bln_rub,source_budget_file,source_liquidity_file
0,2014-02-03,01-02-2014,56508.0,11170.0,803.0,44319.0,216.0,0,NaN,NaN,...,595.3,126.0,0.0,126.0,0.0,3.1,1142.1,825.8,cbr_budget_funds.xlsx,cbr_liquidity.html
1,2014-02-04,01-02-2014,56508.0,11170.0,803.0,44319.0,216.0,0,NaN,NaN,...,601.9,58.7,0.0,58.7,0.0,3.1,1143.8,825.8,cbr_budget_funds.xlsx,cbr_liquidity.html
2,2014-02-05,01-02-2014,56508.0,11170.0,803.0,44319.0,216.0,0,NaN,NaN,...,594.7,62.6,0.0,62.6,0.0,3.1,1215.3,825.8,cbr_budget_funds.xlsx,cbr_liquidity.html


In [5]:
# Convert million to billion
for col in ['budget_funds_total', 'federal_budget_funds',
            'regional_local_budget_funds', 'other_budget_funds',
            'extra_budgetary_funds']:
    src = col + '_mln_rub'
    if src in df.columns:
        df[col + '_bln_rub'] = df[src] / 1000.0

df['treasury_auction_day'] = (df['roskazna_auctions_count'] > 0).astype(int)

df['budget_funds_date_dt'] = pd.to_datetime(df['budget_funds_date'], format='%d-%m-%Y', errors='coerce')
df['budget_funds_lag_days'] = (df['date'] - df['budget_funds_date_dt']).dt.days

print(f"Days with Roskazna auctions: {df['treasury_auction_day'].sum()}")


Days with Roskazna auctions: 1101


In [6]:
# Structural features
df['liquidity_stress'] = -df['liquidity_deficit_surplus_without_correspondent_accounts_bln_rub']
df['net_cbr_position'] = df['cbr_claims_standard_instruments_bln_rub'] - df['cbr_liabilities_standard_instruments_bln_rub']

claims_safe = df['cbr_claims_standard_instruments_bln_rub'].replace(0, np.nan)
df['repo_intensity'] = df['repo_fx_swap_auctions_bln_rub'] / claims_safe

dec2014 = df[df['date'].between('2014-12-01', '2014-12-31')]
print("December 2014 paradox:")
print(f"  surplus RAW: {dec2014['liquidity_deficit_surplus_bln_rub'].max():>8.0f} bln")
print(f"  surplus w/o corr: {dec2014['liquidity_deficit_surplus_without_correspondent_accounts_bln_rub'].max():>8.0f} bln")


December 2014 paradox:
  surplus RAW:     7014 bln
  surplus w/o corr:     7627 bln


In [7]:
# Budget channel - col 14
df['cover_ratio_col14'] = df['roskazna_cover_ratio'].where(df['treasury_auction_day'] == 1)

_cr_s = pd.Series(df['cover_ratio_col14'].values, index=pd.DatetimeIndex(df['date']))
df['cover_ratio_7d_max'] = pd.Series(_cr_s.rolling('7D', min_periods=1).max().values, index=df.index)
df['cover_ratio_weekly_delta'] = df['cover_ratio_col14'].diff(periods=7)

df['budget_funds_bln'] = df['budget_funds_total_bln_rub']
df['budget_funds_7d_delta'] = df['budget_funds_bln'].diff(7)
df['budget_outflow'] = -df['budget_funds_7d_delta']

HIGH_DEMAND = 3.0
df['flag_treasury_stress'] = ((df['treasury_auction_day'] == 1) & (df['cover_ratio_col14'] > HIGH_DEMAND)).astype(int)

auction_df = df[df['treasury_auction_day'] == 1].copy()
print("Col 14 (cover_ratio) on auction days:")
print(auction_df['cover_ratio_col14'].describe().round(3))


Col 14 (cover_ratio) on auction days:
count    1093.000
mean        1.947
std         0.927
min         0.001
25%         1.331
50%         1.748
75%         2.378
max         7.000
Name: cover_ratio_col14, dtype: float64


In [8]:
ROLLING_WINDOW = '1095D'
MAD_MIN = 0.05

def rolling_mad_score(series, dates, window='1095D', mad_min=0.05):
    s = pd.Series(series.values, index=pd.DatetimeIndex(dates))
    median = s.rolling(window, min_periods=1).median()
    mad = (s - median).abs().rolling(window, min_periods=1).median()
    mad = mad.clip(lower=mad_min)
    score = (s - median) / mad
    return pd.Series(score.values, index=series.index)

df['MAD_liquidity_stress'] = rolling_mad_score(df['liquidity_stress'], df['date'])
df['MAD_repo_intensity'] = rolling_mad_score(df['repo_intensity'], df['date'])
df['MAD_net_cbr_position'] = rolling_mad_score(df['net_cbr_position'], df['date'])
df['MAD_budget_delta'] = rolling_mad_score(df['budget_outflow'], df['date'])

cr_ffill = df['cover_ratio_col14'].fillna(method='ffill', limit=7)
df['MAD_cover_ratio_col14'] = rolling_mad_score(cr_ffill, df['date'])

print("MAD-scores OK")


TypeError: NDFrame.fillna() got an unexpected keyword argument 'method'

In [ ]:
CLIP = 5.0
MAD_CORE = ['MAD_liquidity_stress', 'MAD_repo_intensity', 'MAD_net_cbr_position']
MAD_FULL = MAD_CORE + ['MAD_cover_ratio_col14', 'MAD_budget_delta']

for col in MAD_FULL:
    df[col + '_c'] = df[col].clip(-CLIP, CLIP)

df['m5_signal'] = df[[c + '_c' for c in MAD_CORE]].mean(axis=1)
df['m5_signal_full'] = df[[c + '_c' for c in MAD_FULL]].mean(axis=1)

s_min, s_max = df['m5_signal'].min(), df['m5_signal'].max()
df['m5_signal_final'] = (df['m5_signal'] - s_min) / (s_max - s_min) * 10

df['m5_reliable'] = (df['date'] >= df['date'].min() + pd.Timedelta(days=180)).astype(int)

thr75 = df['m5_signal'].quantile(0.75)
df['m5_flag'] = (df['m5_signal'] > thr75).astype(int)

print(f"m5_signal: [{df['m5_signal'].min():.2f}, {df['m5_signal'].max():.2f}]")
print(f"m5_signal_final: [0.00, 10.00]")
print(f"m5_flag (>75p={thr75:.2f}): {df['m5_flag'].sum()} days")
print("\nTop 10 stress days:")
df.nlargest(10, 'm5_signal')[['date', 'm5_signal', 'MAD_liquidity_stress', 'MAD_repo_intensity']]


In [ ]:
# Plot 1 - Liquidity structure
fig, axes = plt.subplots(3, 1, figsize=(16, 13), sharex=True)
fig.suptitle('M5 - Banking Sector Liquidity', fontsize=14)

ax = axes[0]
raw = df['liquidity_deficit_surplus_bln_rub']
clean = df['liquidity_deficit_surplus_without_correspondent_accounts_bln_rub']
ax.fill_between(df.date, raw.clip(upper=0), 0, color='#e74c3c', alpha=0.55, label='Deficit (raw)')
ax.fill_between(df.date, raw.clip(lower=0), 0, color='#2ecc71', alpha=0.35, label='Surplus (raw)')
ax.plot(df.date, clean, color='#f39c12', lw=1.3, label='Without corr accounts')
ax.axhline(0, color='white', lw=0.5, ls='--', alpha=0.4)
shade_crises(ax)
ax.set_ylabel('bln RUB')
ax.set_title('A. Structural position: deficit / surplus', loc='left')
ax.legend(loc='upper left', fontsize=8)
ax.grid(True, alpha=0.35)

ax = axes[1]
ax.stackplot(df.date,
    df['repo_fx_swap_auctions_bln_rub'].fillna(0),
    df['secured_loans_auctions_bln_rub'].fillna(0),
    df['repo_fx_swap_standing_bln_rub'].fillna(0),
    df['secured_loans_standing_bln_rub'].fillna(0),
    labels=['REPO+FX-SWAP auctions', 'Secured loans auctions', 'REPO+FX-SWAP standing', 'Secured loans standing'],
    colors=['#e74c3c','#c0392b','#e67e22','#d35400'], alpha=0.72)
ax2 = ax.twinx()
ax2.plot(df.date, df['repo_intensity'], color='#ffcc00', lw=1.2, ls='--', alpha=0.85)
ax2.set_ylabel('repo_intensity', color='#ffcc00', fontsize=9)
ax2.tick_params(axis='y', colors='#ffcc00')
shade_crises(ax)
ax.set_ylabel('bln RUB')
ax.set_title('B. CBR claims decomposition + repo_intensity (dashed)', loc='left')
ax.legend(loc='upper left', fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)

ax = axes[2]
ax.fill_between(df.date, df['net_cbr_position'], where=df['net_cbr_position'] > 0, color='#e74c3c', alpha=0.6, label='Banks net debtors to CBR')
ax.fill_between(df.date, df['net_cbr_position'], where=df['net_cbr_position'] <= 0, color='#3498db', alpha=0.4, label='CBR net debtor to banks')
ax.axhline(0, color='white', lw=0.5, ls='--', alpha=0.4)
shade_crises(ax)
ax.set_ylabel('bln RUB')
ax.set_xlabel('Date')
ax.set_title('C. Net CBR position = Claims - Liabilities', loc='left')
ax.legend(loc='upper left', fontsize=8)
ax.grid(True, alpha=0.35)

plt.tight_layout()
plt.show()


In [ ]:
FEATURE_COLS = [
    'date',
    'liquidity_deficit_surplus_bln_rub',
    'liquidity_deficit_surplus_without_correspondent_accounts_bln_rub',
    'net_cbr_position',
    'repo_intensity',
    'budget_funds_bln',
    'budget_funds_7d_delta',
    'cover_ratio_col14',
    'cover_ratio_7d_max',
    'cover_ratio_weekly_delta',
    'treasury_auction_day',
    'flag_treasury_stress',
    'MAD_liquidity_stress',
    'MAD_repo_intensity',
    'MAD_net_cbr_position',
    'MAD_cover_ratio_col14',
    'MAD_budget_delta',
    'm5_signal',
    'm5_signal_full',
    'm5_signal_final',
    'm5_flag',
    'm5_reliable',
]

out = df[FEATURE_COLS].copy()
out.to_csv(PROCESSED / 'm5_features.csv', index=False)
print(f"Saved: data/processed/m5_features.csv")
print(f"Rows: {len(out)} | Features: {len(FEATURE_COLS) - 1}")

out[['date', 'm5_signal', 'm5_signal_final', 'm5_reliable']].tail(5)
